# 05. Deep Classification (KoBERT / KoELECTRA)

사전학습 한국어 Transformer 모델을 민원 텍스트 분류에 Fine-tuning합니다.

| Item | Detail |
|------|--------|
| Task | domain(14) / category(63) 분류 |
| Models | KoBERT, KoELECTRA-base-v3 |
| Baseline | 04_baseline_ml (TF-IDF + LR/SVC/LGBM) |
| Primary Metric | Macro F1 Score |
| Environment | Kaggle T4 x2 GPU |
| Visualization | Plotly only |

---
## 0. Environment Setup

In [ ]:
%%capture
!pip install -q plotly kaleido

In [ ]:
import os, json, time, warnings, gc, copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score, confusion_matrix,
)

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'iframe'

warnings.filterwarnings('ignore')

# ---- Kaggle vs Local ----
if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/civilcomplaint-processed'
    OUT_DIR = '/kaggle/working'
    IS_KAGGLE = True
else:
    DATA_DIR = '../data/processed'
    OUT_DIR = '..'
    IS_KAGGLE = False

RESULTS_DIR = os.path.join(OUT_DIR, 'results')
DOMAIN_MODEL_DIR = os.path.join(OUT_DIR, 'models/domain_classifier')
CATEGORY_MODEL_DIR = os.path.join(OUT_DIR, 'models/category_classifier')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DOMAIN_MODEL_DIR, exist_ok=True)
os.makedirs(CATEGORY_MODEL_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count()

# GPU 최적화 설정
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('medium')

print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Device      : {DEVICE} (GPUs: {NUM_GPUS})")
if torch.cuda.is_available():
    print(f"cuDNN bench : enabled")
    print(f"matmul prec : medium")
print(f"Data        : {DATA_DIR}")
print(f"Results     : {RESULTS_DIR}")

In [ ]:
# Load previous ML baseline results
ml_results_path = os.path.join(RESULTS_DIR, 'classification_ml_results.json')
if os.path.exists(ml_results_path):
    with open(ml_results_path) as f:
        ml_results = json.load(f)
    print(f"ML baseline results loaded: {len(ml_results['validation'])} val entries")
    print(f"  Best domain model: {ml_results['best_models']['domain']['model']} "
          f"(Macro F1: {ml_results['best_models']['domain']['macro_f1']})")
else:
    print(f"WARNING: {ml_results_path} not found — will skip baseline comparison")
    ml_results = None

---
## 1. Data Loading

In [ ]:
train_df = pd.read_parquet(f'{DATA_DIR}/train_classification.parquet')
val_df   = pd.read_parquet(f'{DATA_DIR}/val_classification.parquet')
test_df  = pd.read_parquet(f'{DATA_DIR}/test_classification.parquet')

with open(f'{DATA_DIR}/label_mapping.json') as f:
    label_mapping = json.load(f)
with open(f'{DATA_DIR}/class_weights.json') as f:
    class_weights_dict = json.load(f)

NUM_DOMAIN  = len(label_mapping['domain'])
NUM_CATEGORY = len(label_mapping['category'])

print(f"Train: {len(train_df):,} / Val: {len(val_df):,} / Test: {len(test_df):,}")
print(f"Domain classes : {NUM_DOMAIN}")
print(f"Category classes: {NUM_CATEGORY}")
train_df[['text','domain','category']].head(3)

In [ ]:
# Dataset statistics — domain distribution (Plotly)
domain_counts = train_df['domain'].value_counts().sort_values()

fig = go.Figure(go.Bar(
    y=domain_counts.index, x=domain_counts.values, orientation='h',
    marker_color='#42a5f5',
    text=[f'{v:,}' for v in domain_counts.values],
    textposition='outside',
))
fig.update_layout(
    title='Train Set — Domain Distribution',
    xaxis_title='Count', width=800, height=500,
    margin=dict(l=200)
)
fig.show(renderer='iframe')

---
## 2. Tokenizer + Dataset Class

In [ ]:
MAX_LEN = 128
BATCH_SIZE = 64

class ComplaintDataset(Dataset):
    """PyTorch Dataset for complaint text classification."""

    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = texts.tolist() if hasattr(texts, 'tolist') else list(texts)
        self.labels = labels.tolist() if hasattr(labels, 'tolist') else list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
        }


def build_label_encoder(series):
    """Map string labels to integer ids."""
    classes = sorted(series.unique())
    label2id = {c: i for i, c in enumerate(classes)}
    return label2id, {i: c for c, i in label2id.items()}


# Build domain label encoder
domain_label2id, domain_id2label = build_label_encoder(train_df['domain'])
assert len(domain_label2id) == NUM_DOMAIN

y_train_domain = train_df['domain'].map(domain_label2id).values
y_val_domain   = val_df['domain'].map(domain_label2id).values
y_test_domain  = test_df['domain'].map(domain_label2id).values

# Build category label encoder
cat_label2id, cat_id2label = build_label_encoder(train_df['category'])
assert len(cat_label2id) == NUM_CATEGORY

y_train_cat = train_df['category'].map(cat_label2id).values
y_val_cat   = val_df['category'].map(cat_label2id).values
y_test_cat  = test_df['category'].map(cat_label2id).values

print(f"Domain labels  : {NUM_DOMAIN}")
print(f"Category labels: {NUM_CATEGORY}")

In [ ]:
def make_loaders(tokenizer, y_train, y_val, y_test, batch_size=BATCH_SIZE):
    """DataLoader 생성 (pin_memory + persistent_workers 최적화)"""
    train_ds = ComplaintDataset(train_df['text'], y_train, tokenizer)
    val_ds   = ComplaintDataset(val_df['text'],   y_val,   tokenizer)
    test_ds  = ComplaintDataset(test_df['text'],  y_test,  tokenizer)

    use_workers = 2 if IS_KAGGLE else 0
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=use_workers, pin_memory=True,
                              persistent_workers=(use_workers > 0))
    val_loader   = DataLoader(val_ds,   batch_size=batch_size*2, shuffle=False,
                              num_workers=use_workers, pin_memory=True,
                              persistent_workers=(use_workers > 0))
    test_loader  = DataLoader(test_ds,  batch_size=batch_size*2, shuffle=False,
                              num_workers=use_workers, pin_memory=True,
                              persistent_workers=(use_workers > 0))
    return train_loader, val_loader, test_loader

print(f"MAX_LEN={MAX_LEN}, BATCH_SIZE={BATCH_SIZE}")

---
## 3. Training Utilities

In [ ]:
NUM_EPOCHS = 3
LR = 2e-5
WARMUP_RATIO = 0.1
PATIENCE = 2  # Early stopping: 2 epoch 동안 개선 없으면 종료


def train_one_epoch(model, loader, optimizer, scheduler, scaler, device):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * input_ids.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)

        with autocast():
            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)

        total_loss += outputs.loss.item() * input_ids.size(0)
        preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    acc = accuracy_score(all_labels, all_preds)

    return {
        'loss': avg_loss,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'accuracy': acc,
        'preds': all_preds,
        'labels': all_labels,
    }


def train_model(model_name, pretrained_name, num_labels, train_loader,
                val_loader, num_epochs=NUM_EPOCHS, lr=LR, patience=PATIENCE):
    """학습 루프 (Early Stopping + AMP + non_blocking 최적화)"""
    print(f"\n{'='*60}")
    print(f"  Training: {model_name}")
    print(f"  Pretrained: {pretrained_name}")
    print(f"  Classes: {num_labels} | Epochs: {num_epochs} | LR: {lr} | Patience: {patience}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(pretrained_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_name, num_labels=num_labels
    )
    model = model.to(DEVICE)
    if NUM_GPUS > 1:
        model = nn.DataParallel(model)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    scaler = GradScaler()

    history = {'train_loss': [], 'val_loss': [], 'val_macro_f1': [], 'val_weighted_f1': []}
    best_f1 = 0
    best_state = None
    best_metrics = None
    no_improve = 0
    t0 = time.time()

    for epoch in range(1, num_epochs + 1):
        ep_t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, DEVICE)
        val_result = evaluate(model, val_loader, DEVICE)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_result['loss'])
        history['val_macro_f1'].append(val_result['macro_f1'])
        history['val_weighted_f1'].append(val_result['weighted_f1'])

        ep_time = time.time() - ep_t0
        star = ''
        if val_result['macro_f1'] > best_f1:
            best_f1 = val_result['macro_f1']
            raw_model = model.module if hasattr(model, 'module') else model
            best_state = copy.deepcopy(raw_model.state_dict())
            best_metrics = {
                'macro_f1': round(val_result['macro_f1'], 4),
                'weighted_f1': round(val_result['weighted_f1'], 4),
                'accuracy': round(val_result['accuracy'], 4),
            }
            no_improve = 0
            star = ' *best*'
        else:
            no_improve += 1
            star = f' (no improve: {no_improve})'

        print(f"  Epoch {epoch}/{num_epochs}  "
              f"Train Loss={train_loss:.4f}  "
              f"Val Loss={val_result['loss']:.4f}  "
              f"Macro F1={val_result['macro_f1']:.4f}  "
              f"W-F1={val_result['weighted_f1']:.4f}  "
              f"Acc={val_result['accuracy']:.4f}  "
              f"({ep_time:.0f}s){star}")

        # Early Stopping
        if no_improve >= patience:
            print(f"  Early stopping at epoch {epoch} (patience={patience})")
            break

    total_time = time.time() - t0
    print(f"  Total training time: {total_time:.0f}s")
    print(f"  Best Val Macro F1: {best_f1:.4f}")

    return best_state, history, best_metrics, tokenizer


print(f"Training config: epochs={NUM_EPOCHS}, lr={LR}, warmup={WARMUP_RATIO}, patience={PATIENCE}")

---
## 4. KoBERT Fine-tuning (Domain 14-class)

In [ ]:
KOBERT_NAME = 'monologg/kobert'

kobert_tokenizer = AutoTokenizer.from_pretrained(KOBERT_NAME, trust_remote_code=True)
kobert_train_loader, kobert_val_loader, kobert_test_loader = make_loaders(
    kobert_tokenizer, y_train_domain, y_val_domain, y_test_domain
)
print(f"KoBERT tokenizer vocab: {kobert_tokenizer.vocab_size:,}")
print(f"Train batches: {len(kobert_train_loader)} / Val batches: {len(kobert_val_loader)}")

In [ ]:
kobert_state, kobert_history, kobert_val_metrics, _ = train_model(
    model_name='KoBERT',
    pretrained_name=KOBERT_NAME,
    num_labels=NUM_DOMAIN,
    train_loader=kobert_train_loader,
    val_loader=kobert_val_loader,
)

In [ ]:
# KoBERT training curves
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Loss', 'Macro F1'])

epochs_x = list(range(1, NUM_EPOCHS+1))

fig.add_trace(go.Scatter(x=epochs_x, y=kobert_history['train_loss'],
    mode='lines+markers', name='Train Loss', line=dict(color='#42a5f5')), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_x, y=kobert_history['val_loss'],
    mode='lines+markers', name='Val Loss', line=dict(color='#ef5350')), row=1, col=1)

fig.add_trace(go.Scatter(x=epochs_x, y=kobert_history['val_macro_f1'],
    mode='lines+markers', name='Val Macro F1', line=dict(color='#66bb6a')), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs_x, y=kobert_history['val_weighted_f1'],
    mode='lines+markers', name='Val Weighted F1', line=dict(color='#ffa726')), row=1, col=2)

fig.update_layout(title='KoBERT Training Curves (Domain 14-class)',
                  width=900, height=400, margin=dict(t=80))
fig.update_xaxes(title_text='Epoch')
fig.show(renderer='iframe')

In [ ]:
# KoBERT — test evaluation
kobert_model = AutoModelForSequenceClassification.from_pretrained(
    KOBERT_NAME, num_labels=NUM_DOMAIN
)
kobert_model.load_state_dict(kobert_state)
kobert_model = kobert_model.to(DEVICE)

kobert_test_result = evaluate(kobert_model, kobert_test_loader, DEVICE)
print(f"KoBERT Test — Macro F1={kobert_test_result['macro_f1']:.4f}  "
      f"W-F1={kobert_test_result['weighted_f1']:.4f}  "
      f"Acc={kobert_test_result['accuracy']:.4f}")

del kobert_model
gc.collect()
torch.cuda.empty_cache()

---
## 5. KoELECTRA Fine-tuning (Domain 14-class)

In [ ]:
KOELECTRA_NAME = 'monologg/koelectra-base-v3-discriminator'

koelectra_tokenizer = AutoTokenizer.from_pretrained(KOELECTRA_NAME)
koelectra_train_loader, koelectra_val_loader, koelectra_test_loader = make_loaders(
    koelectra_tokenizer, y_train_domain, y_val_domain, y_test_domain
)
print(f"KoELECTRA tokenizer vocab: {koelectra_tokenizer.vocab_size:,}")
print(f"Train batches: {len(koelectra_train_loader)} / Val batches: {len(koelectra_val_loader)}")

In [ ]:
koelectra_state, koelectra_history, koelectra_val_metrics, _ = train_model(
    model_name='KoELECTRA',
    pretrained_name=KOELECTRA_NAME,
    num_labels=NUM_DOMAIN,
    train_loader=koelectra_train_loader,
    val_loader=koelectra_val_loader,
)

In [ ]:
# KoELECTRA training curves
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Loss', 'Macro F1'])

fig.add_trace(go.Scatter(x=epochs_x, y=koelectra_history['train_loss'],
    mode='lines+markers', name='Train Loss', line=dict(color='#42a5f5')), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_x, y=koelectra_history['val_loss'],
    mode='lines+markers', name='Val Loss', line=dict(color='#ef5350')), row=1, col=1)

fig.add_trace(go.Scatter(x=epochs_x, y=koelectra_history['val_macro_f1'],
    mode='lines+markers', name='Val Macro F1', line=dict(color='#66bb6a')), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs_x, y=koelectra_history['val_weighted_f1'],
    mode='lines+markers', name='Val Weighted F1', line=dict(color='#ffa726')), row=1, col=2)

fig.update_layout(title='KoELECTRA Training Curves (Domain 14-class)',
                  width=900, height=400, margin=dict(t=80))
fig.update_xaxes(title_text='Epoch')
fig.show(renderer='iframe')

In [ ]:
# KoELECTRA — test evaluation
koelectra_model = AutoModelForSequenceClassification.from_pretrained(
    KOELECTRA_NAME, num_labels=NUM_DOMAIN
)
koelectra_model.load_state_dict(koelectra_state)
koelectra_model = koelectra_model.to(DEVICE)

koelectra_test_result = evaluate(koelectra_model, koelectra_test_loader, DEVICE)
print(f"KoELECTRA Test — Macro F1={koelectra_test_result['macro_f1']:.4f}  "
      f"W-F1={koelectra_test_result['weighted_f1']:.4f}  "
      f"Acc={koelectra_test_result['accuracy']:.4f}")

del koelectra_model
gc.collect()
torch.cuda.empty_cache()

---
## 6. Comparison Analysis

ML Baseline (LR, SVC, LGBM) vs KoBERT vs KoELECTRA on **domain (14-class)** task.

In [ ]:
# Collect all domain results
dl_val_results = [
    {'model': 'KoBERT', 'level': 'domain', **kobert_val_metrics},
    {'model': 'KoELECTRA', 'level': 'domain', **koelectra_val_metrics},
]
dl_test_results = [
    {'model': 'KoBERT', 'level': 'domain',
     'macro_f1': round(kobert_test_result['macro_f1'], 4),
     'weighted_f1': round(kobert_test_result['weighted_f1'], 4),
     'accuracy': round(kobert_test_result['accuracy'], 4)},
    {'model': 'KoELECTRA', 'level': 'domain',
     'macro_f1': round(koelectra_test_result['macro_f1'], 4),
     'weighted_f1': round(koelectra_test_result['weighted_f1'], 4),
     'accuracy': round(koelectra_test_result['accuracy'], 4)},
]

# Merge with ML baselines
if ml_results is not None:
    ml_domain_val = [r for r in ml_results['validation'] if r['level'] == 'domain']
    ml_domain_test = [r for r in ml_results['test'] if r['level'] == 'domain']
else:
    ml_domain_val = []
    ml_domain_test = []

all_domain_val = ml_domain_val + dl_val_results
all_domain_test = ml_domain_test + dl_test_results

compare_df = pd.DataFrame(all_domain_test)
print("=== Domain Classification — Test Results ===")
compare_df

In [ ]:
# Grouped bar chart — all models on test set
metrics = ['accuracy', 'macro_f1', 'weighted_f1']
metric_labels = ['Accuracy', 'Macro F1', 'Weighted F1']
colors = ['#42a5f5', '#66bb6a', '#ffa726']

fig = go.Figure()
for metric, label, color in zip(metrics, metric_labels, colors):
    fig.add_trace(go.Bar(
        x=compare_df['model'], y=compare_df[metric],
        name=label, marker_color=color,
        text=[f'{v:.4f}' for v in compare_df[metric]],
        textposition='outside',
    ))

fig.update_layout(
    title='Domain Classification — Test Metrics (ML Baseline vs Deep)',
    yaxis_title='Score', yaxis_range=[0, 1.1],
    barmode='group', width=1000, height=500,
    legend=dict(orientation='h', y=1.12),
    margin=dict(t=100),
)
fig.show(renderer='iframe')

In [ ]:
# Per-class F1 horizontal bar chart — best deep model (KoELECTRA)
domain_names = [domain_id2label[i] for i in range(NUM_DOMAIN)]
per_class_f1 = f1_score(
    koelectra_test_result['labels'], koelectra_test_result['preds'],
    average=None, zero_division=0
)
support = np.bincount(koelectra_test_result['labels'], minlength=NUM_DOMAIN)

f1_df = pd.DataFrame({
    'domain': domain_names,
    'f1': per_class_f1,
    'support': support,
}).sort_values('f1')

macro_f1_val = koelectra_test_result['macro_f1']

fig = go.Figure(go.Bar(
    y=f1_df['domain'], x=f1_df['f1'], orientation='h',
    marker_color=[
        '#ef5350' if v < 0.5 else '#ffa726' if v < 0.7 else '#66bb6a'
        for v in f1_df['f1']
    ],
    text=[f"{v:.3f} (n={s:,})" for v, s in zip(f1_df['f1'], f1_df['support'])],
    textposition='outside',
))
fig.add_vline(x=macro_f1_val, line_dash='dash', line_color='red',
              annotation_text=f'Macro F1: {macro_f1_val:.4f}')
fig.update_layout(
    title='Per-Class F1 — Domain (KoELECTRA, Test)',
    xaxis_title='F1', xaxis_range=[0, 1.15],
    width=850, height=550, margin=dict(l=200),
)
fig.show(renderer='iframe')

In [ ]:
# Confusion matrix heatmap — KoELECTRA
cm = confusion_matrix(koelectra_test_result['labels'], koelectra_test_result['preds'])
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig = go.Figure(go.Heatmap(
    z=cm_norm, x=domain_names, y=domain_names, colorscale='Blues',
    text=[[f'{v:.0f}' for v in row] for row in cm],
    texttemplate='%{text}', textfont=dict(size=9),
))
fig.update_layout(
    title='Confusion Matrix — Domain (KoELECTRA, Test)',
    xaxis_title='Predicted', yaxis_title='True',
    width=750, height=650,
    yaxis=dict(autorange='reversed'),
)
fig.show(renderer='iframe')

In [ ]:
# Radar chart — all models comparison
radar_metrics = ['accuracy', 'macro_f1', 'weighted_f1']
radar_labels = ['Accuracy', 'Macro F1', 'Weighted F1']

model_colors = {
    'TF-IDF + LR': '#42a5f5',
    'TF-IDF + SVC': '#66bb6a',
    'TF-IDF + LGBM': '#ffa726',
    'KoBERT': '#ab47bc',
    'KoELECTRA': '#ef5350',
}

fig = go.Figure()
for _, row in compare_df.iterrows():
    values = [row[m] for m in radar_metrics]
    values.append(values[0])  # close the polygon
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_labels + [radar_labels[0]],
        fill='toself',
        name=row['model'],
        line=dict(color=model_colors.get(row['model'], '#888')),
        opacity=0.6,
    ))

fig.update_layout(
    title='Model Comparison — Radar Chart (Domain, Test)',
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    width=700, height=550,
)
fig.show(renderer='iframe')

---
## 7. Category Classification (63-class)

Best domain model (KoELECTRA)을 63-class category 분류에 Fine-tuning합니다.

In [ ]:
cat_tokenizer = AutoTokenizer.from_pretrained(KOELECTRA_NAME)
cat_train_loader, cat_val_loader, cat_test_loader = make_loaders(
    cat_tokenizer, y_train_cat, y_val_cat, y_test_cat
)
print(f"Category classification: {NUM_CATEGORY} classes")
print(f"Train batches: {len(cat_train_loader)}")

In [ ]:
cat_state, cat_history, cat_val_metrics, _ = train_model(
    model_name='KoELECTRA (Category)',
    pretrained_name=KOELECTRA_NAME,
    num_labels=NUM_CATEGORY,
    train_loader=cat_train_loader,
    val_loader=cat_val_loader,
)

In [ ]:
# Category — test evaluation
cat_model = AutoModelForSequenceClassification.from_pretrained(
    KOELECTRA_NAME, num_labels=NUM_CATEGORY
)
cat_model.load_state_dict(cat_state)
cat_model = cat_model.to(DEVICE)

cat_test_result = evaluate(cat_model, cat_test_loader, DEVICE)
print(f"KoELECTRA Category Test — Macro F1={cat_test_result['macro_f1']:.4f}  "
      f"W-F1={cat_test_result['weighted_f1']:.4f}  "
      f"Acc={cat_test_result['accuracy']:.4f}")

del cat_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Category training curves
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Loss', 'Macro F1'])

fig.add_trace(go.Scatter(x=epochs_x, y=cat_history['train_loss'],
    mode='lines+markers', name='Train Loss', line=dict(color='#42a5f5')), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_x, y=cat_history['val_loss'],
    mode='lines+markers', name='Val Loss', line=dict(color='#ef5350')), row=1, col=1)

fig.add_trace(go.Scatter(x=epochs_x, y=cat_history['val_macro_f1'],
    mode='lines+markers', name='Val Macro F1', line=dict(color='#66bb6a')), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs_x, y=cat_history['val_weighted_f1'],
    mode='lines+markers', name='Val Weighted F1', line=dict(color='#ffa726')), row=1, col=2)

fig.update_layout(title='KoELECTRA Training Curves (Category 63-class)',
                  width=900, height=400, margin=dict(t=80))
fig.update_xaxes(title_text='Epoch')
fig.show(renderer='iframe')

---
## 8. Save Results + Base64 Download

In [ ]:
# Save domain classifier
domain_model_save = AutoModelForSequenceClassification.from_pretrained(
    KOELECTRA_NAME, num_labels=NUM_DOMAIN
)
domain_model_save.load_state_dict(koelectra_state)
domain_model_save.save_pretrained(DOMAIN_MODEL_DIR)
koelectra_tokenizer.save_pretrained(DOMAIN_MODEL_DIR)

# Save category classifier
cat_model_save = AutoModelForSequenceClassification.from_pretrained(
    KOELECTRA_NAME, num_labels=NUM_CATEGORY
)
cat_model_save.load_state_dict(cat_state)
cat_model_save.save_pretrained(CATEGORY_MODEL_DIR)
cat_tokenizer.save_pretrained(CATEGORY_MODEL_DIR)

# Copy label_mapping.json to model dirs
import shutil
label_map_src = f'{DATA_DIR}/label_mapping.json'
shutil.copy2(label_map_src, os.path.join(DOMAIN_MODEL_DIR, 'label_mapping.json'))
shutil.copy2(label_map_src, os.path.join(CATEGORY_MODEL_DIR, 'label_mapping.json'))

# Save label2id / id2label mappings
with open(os.path.join(DOMAIN_MODEL_DIR, 'domain_label2id.json'), 'w') as f:
    json.dump(domain_label2id, f, ensure_ascii=False, indent=2)
with open(os.path.join(CATEGORY_MODEL_DIR, 'category_label2id.json'), 'w') as f:
    json.dump(cat_label2id, f, ensure_ascii=False, indent=2)

print(f"Domain model   -> {DOMAIN_MODEL_DIR}")
print(f"Category model -> {CATEGORY_MODEL_DIR}")

del domain_model_save, cat_model_save
gc.collect()

In [ ]:
# Save classification_dl_results.json
dl_results_out = {
    "validation": [
        {'model': 'KoBERT',    'level': 'domain',   **kobert_val_metrics},
        {'model': 'KoELECTRA', 'level': 'domain',   **koelectra_val_metrics},
        {'model': 'KoELECTRA', 'level': 'category', **cat_val_metrics},
    ],
    "test": [
        {'model': 'KoBERT', 'level': 'domain',
         'macro_f1': round(kobert_test_result['macro_f1'], 4),
         'weighted_f1': round(kobert_test_result['weighted_f1'], 4),
         'accuracy': round(kobert_test_result['accuracy'], 4)},
        {'model': 'KoELECTRA', 'level': 'domain',
         'macro_f1': round(koelectra_test_result['macro_f1'], 4),
         'weighted_f1': round(koelectra_test_result['weighted_f1'], 4),
         'accuracy': round(koelectra_test_result['accuracy'], 4)},
        {'model': 'KoELECTRA', 'level': 'category',
         'macro_f1': round(cat_test_result['macro_f1'], 4),
         'weighted_f1': round(cat_test_result['weighted_f1'], 4),
         'accuracy': round(cat_test_result['accuracy'], 4)},
    ],
    "best_models": {
        "domain": {
            'model': 'KoELECTRA',
            'pretrained': KOELECTRA_NAME,
            'level': 'domain',
            'macro_f1': round(koelectra_test_result['macro_f1'], 4),
            'weighted_f1': round(koelectra_test_result['weighted_f1'], 4),
            'accuracy': round(koelectra_test_result['accuracy'], 4),
        },
        "category": {
            'model': 'KoELECTRA',
            'pretrained': KOELECTRA_NAME,
            'level': 'category',
            'macro_f1': round(cat_test_result['macro_f1'], 4),
            'weighted_f1': round(cat_test_result['weighted_f1'], 4),
            'accuracy': round(cat_test_result['accuracy'], 4),
        },
    }
}

dl_results_path = os.path.join(RESULTS_DIR, 'classification_dl_results.json')
with open(dl_results_path, 'w') as f:
    json.dump(dl_results_out, f, ensure_ascii=False, indent=2)

print(f"Results -> {dl_results_path}")
pd.DataFrame(dl_results_out['test'])

In [ ]:
# Base64 download helpers
import base64, zipfile, io
from IPython.display import display, HTML


def create_download_link(filepath, filename=None):
    if filename is None:
        filename = filepath.split('/')[-1]
    with open(filepath, 'rb') as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    href = (f'<a href="data:application/octet-stream;base64,{b64}" '
            f'download="{filename}">\U0001F4E5 {filename} ({len(data)/1024/1024:.1f} MB)</a>')
    display(HTML(href))


def create_zip_download(file_dict, zip_name="model_artifacts.zip"):
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
        for arcname, filepath in file_dict.items():
            zf.write(filepath, arcname)
    buffer.seek(0)
    b64 = base64.b64encode(buffer.read()).decode()
    size_mb = buffer.tell() / 1024 / 1024
    href = (f'<a href="data:application/octet-stream;base64,{b64}" '
            f'download="{zip_name}">\U0001F4E6 {zip_name} ({size_mb:.1f} MB)</a>')
    display(HTML(href))


print("Download helpers ready.")

In [ ]:
# Download results JSON
create_download_link(dl_results_path, 'classification_dl_results.json')

In [ ]:
# Download domain classifier as zip
domain_files = {}
for fname in os.listdir(DOMAIN_MODEL_DIR):
    fpath = os.path.join(DOMAIN_MODEL_DIR, fname)
    if os.path.isfile(fpath):
        domain_files[f'domain_classifier/{fname}'] = fpath

create_zip_download(domain_files, 'domain_classifier.zip')

# Download category classifier as zip
cat_files = {}
for fname in os.listdir(CATEGORY_MODEL_DIR):
    fpath = os.path.join(CATEGORY_MODEL_DIR, fname)
    if os.path.isfile(fpath):
        cat_files[f'category_classifier/{fname}'] = fpath

create_zip_download(cat_files, 'category_classifier.zip')

---
## 9. Summary

In [ ]:
# Progressive improvement visualization
progress_models = []

if ml_results is not None:
    for r in ml_results['test']:
        if r['level'] == 'domain':
            progress_models.append(r)

progress_models.extend(dl_test_results)
progress_df = pd.DataFrame(progress_models)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=progress_df['model'], y=progress_df['macro_f1'],
    marker_color=['#90caf9', '#a5d6a7', '#ffe082', '#ce93d8', '#ef9a9a'],
    text=[f'{v:.4f}' for v in progress_df['macro_f1']],
    textposition='outside',
))
fig.update_layout(
    title='Progressive Improvement — Domain Macro F1 (Test)',
    yaxis_title='Macro F1', yaxis_range=[0, 1.0],
    width=800, height=450,
    shapes=[dict(type='line', x0=-0.5, x1=len(progress_df)-0.5,
                 y0=progress_df['macro_f1'].max(), y1=progress_df['macro_f1'].max(),
                 line=dict(color='red', dash='dash'))],
)
fig.show(renderer='iframe')

In [ ]:
print("=" * 60)
print("     05. Deep Classification — Summary")
print("=" * 60)

print("\n[Domain 14-class — Test]")
for r in all_domain_test:
    print(f"  {r['model']:<18} Macro F1={r['macro_f1']:.4f}  "
          f"W-F1={r['weighted_f1']:.4f}  Acc={r['accuracy']:.4f}")

print(f"\n[Category 63-class — Test]")
print(f"  {'KoELECTRA':<18} Macro F1={cat_test_result['macro_f1']:.4f}  "
      f"W-F1={cat_test_result['weighted_f1']:.4f}  "
      f"Acc={cat_test_result['accuracy']:.4f}")

print(f"\nBest Domain   : KoELECTRA (Macro F1={koelectra_test_result['macro_f1']:.4f})")
print(f"Best Category : KoELECTRA (Macro F1={cat_test_result['macro_f1']:.4f})")

print(f"\nArtifacts:")
print(f"  Domain model   -> {DOMAIN_MODEL_DIR}")
print(f"  Category model -> {CATEGORY_MODEL_DIR}")
print(f"  Results JSON   -> {dl_results_path}")

print(f"\nNext -> 06_bm25_retrieval")
print("=" * 60)

---
## 10. 모델 선택 및 하이퍼파라미터 근거

### 모델 선택

| Model | Base Architecture | Why Selected |
|-------|------------------|--------------|
| **KoBERT** | BERT (Masked LM) | 한국어 특화 사전학습, 한국어 NLP 커뮤니티에서 가장 폭넓게 검증된 baseline |
| **KoELECTRA** | ELECTRA (Replaced Token Detection) | BERT 대비 학습 효율 4배 향상 (RTD pretraining). 한국어 분류 task에서 일관적으로 KoBERT 대비 우위 보고 |

ELECTRA의 RTD(Replaced Token Detection)는 모든 토큰에 대해 "진짜/가짜" 판별을 학습하므로, MLM(15% 마스킹)보다 토큰당 학습 신호가 풍부합니다. 이는 특히 **짧은 민원 텍스트** 분류에서 효과적입니다.

### 주요 하이퍼파라미터

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `MAX_LEN` | 128 | 전처리된 민원 텍스트의 95th percentile이 약 110 토큰. 128은 대부분의 샘플을 커버하면서 메모리 효율적 |
| `BATCH_SIZE` | 64 | T4 GPU 2장 × 128 토큰 기준 GPU 메모리 활용 최적화. DataParallel 분배 후 GPU당 32 |
| `LR` | 2e-5 | Transformer fine-tuning의 표준 학습률 (Devlin et al., 2019). 사전학습 가중치를 파괴하지 않는 수준 |
| `NUM_EPOCHS` | 3 | 분류 fine-tuning은 2–4 epoch이 최적 (과적합 방지). Validation 기준 early stopping 적용 |
| `WARMUP_RATIO` | 0.1 | 전체 step의 10%를 warmup으로 사용. 초기 불안정한 gradient를 완화 |

In [ ]:
# === ML Baseline → Deep Learning 개선율 분석 ===
print("=" * 65)
print("  ML Baseline → Deep Learning 개선율 (Domain Classification)")
print("=" * 65)

if ml_results is not None:
    ml_best = ml_results['best_models']['domain']
    ml_f1 = ml_best['macro_f1']
    dl_f1 = koelectra_test_result['macro_f1']

    abs_delta = dl_f1 - ml_f1
    rel_delta = (abs_delta / ml_f1) * 100 if ml_f1 > 0 else 0

    print(f"\n  {'Model':<22} {'Macro F1':>10}")
    print(f"  {'-'*34}")
    print(f"  {'ML Best (' + ml_best['model'] + ')':<22} {ml_f1:>10.4f}")
    print(f"  {'KoELECTRA':<22} {dl_f1:>10.4f}")
    print(f"  {'-'*34}")
    print(f"  {'Absolute Delta':<22} {abs_delta:>+10.4f}")
    print(f"  {'Relative Improvement':<22} {rel_delta:>+9.1f}%")

    # Per-metric comparison
    ml_acc = ml_best.get('accuracy', 0)
    dl_acc = koelectra_test_result['accuracy']
    ml_wf1 = ml_best.get('weighted_f1', 0)
    dl_wf1 = koelectra_test_result['weighted_f1']

    print(f"\n  {'Metric':<18} {'ML Best':>10} {'KoELECTRA':>10} {'Delta':>10}")
    print(f"  {'-'*50}")
    print(f"  {'Macro F1':<18} {ml_f1:>10.4f} {dl_f1:>10.4f} {dl_f1-ml_f1:>+10.4f}")
    print(f"  {'Weighted F1':<18} {ml_wf1:>10.4f} {dl_wf1:>10.4f} {dl_wf1-ml_wf1:>+10.4f}")
    print(f"  {'Accuracy':<18} {ml_acc:>10.4f} {dl_acc:>10.4f} {dl_acc-ml_acc:>+10.4f}")
else:
    print("  ML baseline results not available for comparison.")

print("=" * 65)

---
## 11. Cross-Stage 연결: Classification → Retrieval

**분류(Classification) 결과가 검색(Retrieval)에 미치는 영향:**

1. **도메인 필터링**: 분류기가 예측한 도메인으로 검색 범위를 제한 → 검색 정확도 향상 + 속도 개선
   - 예: "도로 파손" 민원 → `교통` 도메인 QA만 검색 (전체 57K → 도메인별 ~4K)
2. **카테고리 가중**: 63개 카테고리 정보를 검색 쿼리 확장(query expansion)에 활용 가능
3. **오분류 영향**: Domain 분류 오류 → 검색 대상 자체가 잘못됨 → **분류 성능이 전체 파이프라인의 상한선**

**다음 단계 (06_bm25_retrieval):**
- 분류된 도메인 내에서 키워드 기반 검색(BM25) baseline 구축
- Domain-specific vs Global retrieval 성능 비교 예정

In [ ]:
# === ML vs DL 도메인별 F1 Delta 분석 ===
import joblib

ML_MODELS_DIR = os.path.join(OUT_DIR, 'models/classification/ml_baseline')

ml_per_class_f1 = None
ml_model_name = 'ML Best'

try:
    _tfidf = joblib.load(os.path.join(ML_MODELS_DIR, 'tfidf_vectorizer.joblib'))
    _le_dict = joblib.load(os.path.join(ML_MODELS_DIR, 'label_encoders.joblib'))
    _ml_model = joblib.load(os.path.join(ML_MODELS_DIR, 'best_domain.joblib'))

    _X_test = _tfidf.transform(test_df['text'])
    _le_d = _le_dict['domain']
    _y_test = _le_d.transform(test_df['domain'])
    _ml_pred = _ml_model.predict(_X_test)
    ml_per_class_f1_raw = f1_score(_y_test, _ml_pred, average=None, zero_division=0)

    # DL 도메인 순서에 맞춰 정렬
    ml_domain_map = {name: f1 for name, f1 in zip(_le_d.classes_, ml_per_class_f1_raw)}
    ml_per_class_f1 = np.array([ml_domain_map.get(domain_id2label[i], 0) for i in range(NUM_DOMAIN)])
    ml_model_name = ml_results['best_models']['domain']['model'] if ml_results else 'ML Best'

    del _tfidf, _le_dict, _ml_model, _X_test
    gc.collect()
    print(f"ML 모델 로드 완료: {ml_model_name}")
except Exception as e:
    print(f"ML 모델 로드 불가 ({e}) — 전체 지표 비교만 수행")

if ml_per_class_f1 is not None:
    # 도메인별 delta: DL - ML
    delta_f1 = per_class_f1 - ml_per_class_f1

    delta_df = pd.DataFrame({
        'domain': [domain_id2label[i] for i in range(NUM_DOMAIN)],
        'ml_f1': ml_per_class_f1,
        'dl_f1': per_class_f1,
        'delta': delta_f1,
    }).sort_values('delta', ascending=True)

    colors = ['#66bb6a' if d > 0 else '#ef5350' for d in delta_df['delta']]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=delta_df['domain'],
        x=delta_df['delta'],
        orientation='h',
        marker_color=colors,
        text=[f'{d:+.3f} ({ml:.3f} -> {dl:.3f})' for d, ml, dl
              in zip(delta_df['delta'], delta_df['ml_f1'], delta_df['dl_f1'])],
        textposition='outside',
    ))

    fig.add_vline(x=0, line_color='black', line_width=1)

    avg_delta = delta_df['delta'].mean()
    fig.add_vline(x=avg_delta, line_dash='dash', line_color='blue',
                  annotation_text=f'평균 Delta: {avg_delta:+.3f}')

    fig.update_layout(
        title=f'도메인별 F1 Delta: {ml_model_name} → KoELECTRA',
        xaxis_title='F1 Delta (DL - ML)',
        width=900, height=550,
        margin=dict(l=150),
    )
    fig.show(renderer='iframe')

    print(f"\n평균 도메인별 F1 Delta: {avg_delta:+.4f}")
    print(f"최대 개선: {delta_df.iloc[-1]['domain']} (Delta={delta_df.iloc[-1]['delta']:+.4f})")
    print(f"최소 개선: {delta_df.iloc[0]['domain']} (Delta={delta_df.iloc[0]['delta']:+.4f})")

    improved = (delta_df['delta'] > 0).sum()
    print(f"\n개선된 도메인: {improved}/{NUM_DOMAIN}")
else:
    print("도메인별 delta 차트 생략 (ML 모델 미사용)")
    print("전체 delta 비교는 위 Section 10 참조")

---
## 12. ML vs DL 도메인별 성능 차이 분석

### DL이 가장 효과적인 도메인

- **소수 클래스**: 학습 샘플이 적은 도메인(부동산, 숙박 등)에서 KoELECTRA가 가장 큰 개선폭을 보임
  - 이유: 사전학습된 한국어 지식이 소수 클래스의 feature 부족을 보완
- **유사 도메인 구분**: `부동산` vs `부동산업`처럼 TF-IDF가 구분하지 못하는 도메인에서 DL이 문맥 기반으로 구분

### DL이 ML 대비 개선되지 않는 도메인

- **고빈도 도메인** (K쇼핑, 질병관리본부): 이미 ML에서 높은 F1 달성 → DL의 marginal gain이 작음
  - 원인 가설: 충분한 학습 데이터가 있으면 TF-IDF의 bag-of-words 표현만으로도 높은 구분력 확보
- **단순 키워드 도메인**: 특정 키워드만으로 명확히 구분 가능한 도메인은 TF-IDF와 Transformer 차이가 미미

### 시사점

DL 모델의 가치는 **소수 클래스 일반화**와 **유사 도메인 구분**에 있음. 고빈도 도메인만 대상으로 한다면 ML baseline으로 충분하지만, 전체 14개 도메인의 **균등한 성능** (Macro F1)이 목표이므로 DL 모델 선택이 합리적입니다.

In [ ]:
# === Kaggle Dataset 자동 업로드 ===
# 다음 노트북(06~08)에서 분류 모델 + 결과를 입력으로 사용
if IS_KAGGLE:
    UPLOAD_DIR = '/kaggle/working/dataset_upload'
    os.makedirs(UPLOAD_DIR, exist_ok=True)

    # 심볼릭 링크 생성 (파일 복사 대신 메모리 절약)
    upload_dirs = [DOMAIN_MODEL_DIR, CATEGORY_MODEL_DIR]
    for model_dir in upload_dirs:
        if os.path.exists(model_dir):
            dir_name = os.path.basename(model_dir)
            dst_dir = os.path.join(UPLOAD_DIR, dir_name)
            if not os.path.exists(dst_dir):
                os.symlink(model_dir, dst_dir)

    # 결과 JSON
    results_json = os.path.join(RESULTS_DIR, 'classification_dl_results.json')
    dst_json = os.path.join(UPLOAD_DIR, 'classification_dl_results.json')
    if os.path.exists(results_json) and not os.path.exists(dst_json):
        os.symlink(results_json, dst_json)

    meta = {
        "title": "civilcomplaint-dl-classification",
        "id": "kukass/civilcomplaint-dl-classification",
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(f'{UPLOAD_DIR}/dataset-metadata.json', 'w') as f:
        json.dump(meta, f)

    print(f"업로드 대상: {os.listdir(UPLOAD_DIR)}")
    !kaggle datasets create -p {UPLOAD_DIR} --dir-mode zip
    print("Kaggle 데이터셋 업로드 완료: civilcomplaint-dl-classification")
else:
    print("로컬 환경 — Kaggle 업로드 생략")